#Name - Pratik Pradeep Rughe

# Option B : Duplicate Detection in the SSH Open Marketplace

## Introduction

This notebook addresses **Option B: Duplicate detection** from the Research Software Engineer / Technical Developer assessment.

The SSH Open Marketplace brings together tools and services from many different sources. Because of this, it’s quite common for the same resource to appear more than once, sometimes with slightly different names, descriptions, or metadata.

In this notebook, I focus on the **tool-or-service category** and explore how we can identify these duplicate or near-duplicate entries in a practical way.

Rather than trying to build a fully automated system, I take a step-by-step approach. I start with stronger and more reliable signals, such as URLs and identifiers, and then use fuzzy matching on titles to catch cases where the wording is slightly different. The goal is to keep the method simple, transparent, and easy to explain.

In the end, instead of making automatic decisions, the workflow produces a small set of likely duplicate pairs along with supporting evidence. This makes it easier for a human reviewer to quickly inspect and decide which records should be merged or cleaned up.

## Environment notes

This notebook is designed to run end-to-end in a standard Python environment.

Required packages:

```text
requests
pandas
rapidfuzz
```

The notebook also uses Python standard-library modules: `re`, `itertools`, and `urllib.parse`.


## 1. Setup

In this section, I import the libraries needed for the task, including tools for calling the API, working with data, cleaning text, comparing records, and matching similar strings. I also adjust a few display settings so that longer text fields are shown properly, making it easier to read and inspect the data directly in the notebook.


In [ ]:
import requests  # for making API calls
import pandas as pd  # for working with tabular data
import re  # for text cleaning
import itertools  # for generating item pairs
from rapidfuzz import fuzz  # for fuzzy title matching
from urllib.parse import urlparse  # for URL parsing and normalization

pd.set_option("display.max_colwidth", 200)  # show longer text values
pd.set_option("display.max_columns", None)  # show all dataframe columns

## 2. Fetch tool-or-service records from the API



The assessment asks for a meaningful sample from the **tool-or-service** category, so I use pagination to collect the data. Specifically, I retrieve five pages with 100 records per page, which gives a working sample of up to 500 records.

To keep things organized, I split the logic into two small helper functions. The function `fetch_page` handles a single API request, while `fetch_tools_services` loops through multiple pages and combines all the results into one list. This makes the data collection step easier to manage and reuse.


In [ ]:
BASE_URL = "https://marketplace-api.sshopencloud.eu/api"  # base API URL

def fetch_page(endpoint, params=None):
    """Fetch one page from a Marketplace API endpoint."""
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params=params or {},
        timeout=30
    )
    response.raise_for_status()  # raise an error if the request fails
    return response.json()

def fetch_tools_services(max_pages=5, perpage=100):
    """Fetch a paginated sample of tool-or-service records."""
    all_items = []

    for page in range(1, max_pages + 1):
        data = fetch_page(
            "tools-services",
            params={"page": page, "perpage": perpage}
        )

        page_items = data.get("tools", [])

        if not page_items:
            print(f"No items found on page {page}. Stopping.")
            break

        all_items.extend(page_items)
        print(f"Fetched page {page}: {len(page_items)} tools")

    return all_items

## 3. Preparing the dataset

The API response is nested JSON, so I normalize it into a pandas DataFrame. I then keep only the metadata fields that are useful for duplicate detection:

- `label`: the tool/service title
- `description`: supporting text
- `persistentId`: Marketplace-level identifier
- `externalIds`: identifiers from external systems
- `accessibleAt`: URLs where the resource can be accessed
- `sourceItemId`: identifier in the original source
- `source.label` and `source.id`: source information

Keeping a smaller set of fields makes the workflow easier to explain and review.


In [ ]:
items = fetch_tools_services(max_pages=5, perpage=100)
df = pd.json_normalize(items)

# Expected columns used by the deduplication workflow
selected_columns = [
    "id",
    "label",
    "description",
    "persistentId",
    "externalIds",
    "accessibleAt",
    "sourceItemId",
    "source.label",
    "source.id",
]

# Reindex avoids errors if an optional field is missing in some API responses
dup_df = df.reindex(columns=selected_columns).copy()

print("Total items:", len(dup_df))
dup_df.head()

Fetched page 1: 100 tools
Fetched page 2: 100 tools
Fetched page 3: 100 tools
Fetched page 4: 100 tools
Fetched page 5: 100 tools
Total items: 500


,id,label,description,persistentId,externalIds,accessibleAt,sourceItemId,source.label,source.id
0,72078,140kit,"140kit provides a management layer for tweet collection and analysis.\n\nRaw data cannot be passed through to the users, but any analytical process can be run across your dataset, and the data is ...",SIU1nO,[],[https://github.com/WebEcologyProject/140kit],937,TAPoR,1.0
1,81603,1641 Depositions - Trinity College Dublin,"Fully searchable digital edition of the 1641 Depositions at Trinity College Dublin Library, comprising transcripts and images of all 8,000 depositions, examinations and associated materials in whi...",i60Tk3,[],[https://1641.tcd.ie/],NaN,NaN,NaN
2,72079,360 stopni: 360-degree documentation,"Visit, without leaving home.\nA service involving 360-degree documentation and making the results available in the form of a virtual walk-through or virtual gallery. Aimed primarily at researchers...",tZUwaR,[],[https://lab.dariah.pl],127,Dariah.lab Poland,452.0
3,75428,3D fotogrametria: 3D models of any objects (photogrammetric method),"Everything can be scanned\nThe photogrammetric method makes it possible to make 3D models of various objects, from small objects (e.g. archaeological artefacts) to entire architectural complexes. ...",tLdtav,[],[https://analytics.umcs.pl/],134,Dariah.lab Poland,452.0
4,36324,3DF Zephyr - photogrammetry software - 3d models from photos,"3DF Zephyr\[1\]\[2\] is a commercial photogrammetry and 3D modeling software. Developed and marketed by the Italian software house 3DFLOW, 3DF Zephyr was first released in January 2014 and continu...",4gDAHv,[],[https://www.3dflow.net/3df-zephyr-pro-3d-models-from-photos/],WQFP6XPS,SSK Zotero Resources,3.0


## 4. Normalize text, URLs, and identifiers

Duplicate records are often not identical at the raw metadata level. For example, the same URL may appear with or without `www`, and titles may differ only in capitalization or punctuation.

To make comparisons more reliable, I define helper functions to:

- normalize text labels
- extract the first available URL
- normalize URLs by removing `www` and trailing slashes
- extract external identifiers from nested metadata


In [ ]:
def normalize_text(text):
    """Normalize labels/descriptions for comparison."""
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)  # remove punctuation/special characters
    text = re.sub(r"\s+", " ", text)  # collapse repeated spaces

    return text.strip()

def first_url(value):
    """Extract the first URL-like value from a string, list, or dictionary."""
    if isinstance(value, str):
        return value

    if isinstance(value, list) and len(value) > 0:
        first = value[0]

        if isinstance(first, str):
            return first

        if isinstance(first, dict):
            return first.get("url") or first.get("label") or ""

    if isinstance(value, dict):
        return value.get("url") or value.get("label") or ""

    return ""

def normalize_url(url):
    """Normalize a URL so equivalent URLs are easier to match."""
    if not isinstance(url, str) or not url.strip():
        return ""

    parsed = urlparse(url.strip())
    netloc = parsed.netloc.lower().replace("www.", "")
    path = parsed.path.rstrip("/")

    return f"{netloc}{path}"

def extract_external_id(value):
    """Extract the first external identifier if available."""
    if isinstance(value, list) and len(value) > 0:
        first = value[0]

        if isinstance(first, dict):
            return first.get("identifier", "")

    return

## 5. Create normalized comparison columns

I keep the original metadata as it is, and add a few new normalized columns alongside it for comparison. These cleaned versions of the data make it easier to check for exact duplicates and also support fuzzy matching when looking for near-duplicate records.


In [ ]:
dup_df["label_norm"] = dup_df["label"].apply(normalize_text)
dup_df["description_norm"] = dup_df["description"].apply(normalize_text)
dup_df["accessibleAt_url"] = dup_df["accessibleAt"].apply(first_url)
dup_df["accessibleAt_norm"] = dup_df["accessibleAt_url"].apply(normalize_url)
dup_df["external_id"] = dup_df["externalIds"].apply(extract_external_id)

dup_df.head()

,id,label,description,persistentId,externalIds,accessibleAt,sourceItemId,source.label,source.id,label_norm,description_norm,accessibleAt_url,accessibleAt_norm,external_id
0,72078,140kit,"140kit provides a management layer for tweet collection and analysis.\n\nRaw data cannot be passed through to the users, but any analytical process can be run across your dataset, and the data is ...",SIU1nO,[],[https://github.com/WebEcologyProject/140kit],937,TAPoR,1.0,140kit,140kit provides a management layer for tweet collection and analysis raw data cannot be passed through to the users but any analytical process can be run across your dataset and the data is held f...,https://github.com/WebEcologyProject/140kit,github.com/WebEcologyProject/140kit,None
1,81603,1641 Depositions - Trinity College Dublin,"Fully searchable digital edition of the 1641 Depositions at Trinity College Dublin Library, comprising transcripts and images of all 8,000 depositions, examinations and associated materials in whi...",i60Tk3,[],[https://1641.tcd.ie/],NaN,NaN,NaN,1641 depositions trinity college dublin,fully searchable digital edition of the 1641 depositions at trinity college dublin library comprising transcripts and images of all 8 000 depositions examinations and associated materials in which...,https://1641.tcd.ie/,1641.tcd.ie,None
2,72079,360 stopni: 360-degree documentation,"Visit, without leaving home.\nA service involving 360-degree documentation and making the results available in the form of a virtual walk-through or virtual gallery. Aimed primarily at researchers...",tZUwaR,[],[https://lab.dariah.pl],127,Dariah.lab Poland,452.0,360 stopni 360 degree documentation,visit without leaving home a service involving 360 degree documentation and making the results available in the form of a virtual walk through or virtual gallery aimed primarily at researchers and...,https://lab.dariah.pl,lab.dariah.pl,None
3,75428,3D fotogrametria: 3D models of any objects (photogrammetric method),"Everything can be scanned\nThe photogrammetric method makes it possible to make 3D models of various objects, from small objects (e.g. archaeological artefacts) to entire architectural complexes. ...",tLdtav,[],[https://analytics.umcs.pl/],134,Dariah.lab Poland,452.0,3d fotogrametria 3d models of any objects photogrammetric method,everything can be scanned the photogrammetric method makes it possible to make 3d models of various objects from small objects e g archaeological artefacts to entire architectural complexes models...,https://analytics.umcs.pl/,analytics.umcs.pl,None
4,36324,3DF Zephyr - photogrammetry software - 3d models from photos,"3DF Zephyr\[1\]\[2\] is a commercial photogrammetry and 3D modeling software. Developed and marketed by the Italian software house 3DFLOW, 3DF Zephyr was first released in January 2014 and continu...",4gDAHv,[],[https://www.3dflow.net/3df-zephyr-pro-3d-models-from-photos/],WQFP6XPS,SSK Zotero Resources,3.0,3df zephyr photogrammetry software 3d models from photos,3df zephyr 1 2 is a commercial photogrammetry and 3d modeling software developed and marketed by the italian software house 3dflow 3df zephyr was first released in january 2014 and continuously up...,https://www.3dflow.net/3df-zephyr-pro-3d-models-from-photos/,3dflow.net/3df-zephyr-pro-3d-models-from-photos,None


## 6. Check exact duplicate signals

Before moving on to fuzzy matching, I first look at stronger and more reliable signals. Exact matches on labels, normalized labels, URLs, and external identifiers usually give a clearer indication of duplicates than title similarity alone.

Looking at these counts provides a quick first impression of where duplicates might exist in the sample, and helps guide the next steps in the analysis.


In [ ]:
exact_label_dups = dup_df[
    dup_df.duplicated("label", keep=False)
].sort_values("label")

normalized_label_dups = dup_df[
    dup_df.duplicated("label_norm", keep=False)
].sort_values("label_norm")

url_dups = dup_df[
    (dup_df["accessibleAt_norm"] != "") &
    dup_df.duplicated("accessibleAt_norm", keep=False)
].sort_values("accessibleAt_norm")

external_dups = dup_df[
    (dup_df["external_id"] != "") &
    dup_df.duplicated("external_id", keep=False)
].sort_values("external_id")

print("Exact label duplicates:", len(exact_label_dups))
print("Normalized label duplicates:", len(normalized_label_dups))
print("Shared URL duplicates:", len(url_dups))
print("Shared external ID duplicates:", len(external_dups))

Exact label duplicates: 2
Normalized label duplicates: 8
Shared URL duplicates: 31
Shared external ID duplicates: 434


## 7. Generate fuzzy duplicate candidates

Some duplicates or near-duplicate records won’t have exactly the same title, so relying only on exact matches isn’t enough. To catch these cases, I compare the normalized labels using fuzzy string matching, which helps identify similar titles even when the wording is slightly different.

To keep things efficient, I also apply a simple filtering step before comparing every pair. If two titles are very different in length, I skip comparing them. This reduces unnecessary work while still allowing genuinely similar titles to be matched.


In [ ]:
candidate_pairs = []

subset = dup_df[dup_df["label_norm"].str.len() > 3].reset_index(drop=True)

for i, j in itertools.combinations(range(len(subset)), 2):
    a = subset.loc[i]
    b = subset.loc[j]

    # Simple blocking rule to avoid comparing clearly unrelated titles
    if abs(len(a["label_norm"]) - len(b["label_norm"])) > 15:
        continue

    score = fuzz.token_sort_ratio(a["label_norm"], b["label_norm"])

    if score >= 90:
        candidate_pairs.append({
            "id_1": a["id"],
            "label_1": a["label"],
            "source_1": a["source.label"],
            "id_2": b["id"],
            "label_2": b["label"],
            "source_2": b["source.label"],
            "title_score": score,
            "same_external_id": (
                a["external_id"] != "" and a["external_id"] == b["external_id"]
            ),
            "same_url": (
                a["accessibleAt_norm"] != "" and a["accessibleAt_norm"] == b["accessibleAt_norm"]
            ),
            "same_persistentId": (
                pd.notna(a["persistentId"]) and
                pd.notna(b["persistentId"]) and
                a["persistentId"] == b["persistentId"]
            ),
            "same_sourceItemId": (
                pd.notna(a["sourceItemId"]) and
                pd.notna(b["sourceItemId"]) and
                a["sourceItemId"] == b["sourceItemId"] and
                a["source.label"] == b["source.label"]
            )
        })

pairs_df = pd.DataFrame(candidate_pairs)

print("Candidate pairs found:", len(pairs_df))
pairs_df.head()

Candidate pairs found: 12


,id_1,label_1,source_1,id_2,label_2,source_2,title_score,same_external_id,same_url,same_persistentId,same_sourceItemId
0,82442,Alpino-Webservice,CLARIAH-NL Tools,82443,Alpino-Webservice,CLARIAH-NL Tools,100.000000,True,True,False,True
1,67391,A.nnotate,TAPoR,67392,Annotate,TAPoR,94.117647,True,False,False,False
2,75999,Antconc,CLARIN Resource Families,78153,AntConc,NaN,100.000000,True,True,False,False
3,75999,Antconc,CLARIN Resource Families,76000,AntPConc,CLARIN Resource Families,93.333333,True,False,False,False
4,78153,AntConc,NaN,76000,AntPConc,CLARIN Resource Families,93.333333,True,False,False,False


## 8. Assign confidence levels

Not every candidate pair should be treated as a definite duplicate. To make the results more useful, I assign a confidence level to each pair based on how strong the supporting signals are.

The idea is to keep the rules simple and easy to understand:

**High confidence**: cases where there is strong evidence, such as a shared identifier, the same persistent ID, the same URL combined with high title similarity, or almost identical titles.

**Medium confidence**: cases with good title similarity or matching source item IDs, but without stronger supporting metadata.

**Low confidence**: cases where the evidence is weak, which are not included in the final reviewer table

This way, the output focuses on the most relevant pairs and makes it easier for a human reviewer to decide what to do next.


In [ ]:
def classify_pair(row):
    """Classify a candidate pair based on available duplicate signals."""
    if row["same_external_id"]:
        return "high"

    if row["same_persistentId"]:
        return "high"

    if row["same_url"] and row["title_score"] >= 85:
        return "high"

    if row["same_sourceItemId"]:
        return "medium"

    if row["title_score"] >= 96:
        return "high"

    if row["title_score"] >= 90:
        return "medium"

    return "low"

if not pairs_df.empty:
    pairs_df["confidence"] = pairs_df.apply(classify_pair, axis=1)
    pairs_df = pairs_df.sort_values(
        by=["confidence", "title_score"],
        ascending=[True, False]
    )
else:
    pairs_df["confidence"] = pd.Series(dtype="object")

pairs_df.head(20)

,id_1,label_1,source_1,id_2,label_2,source_2,title_score,same_external_id,same_url,same_persistentId,same_sourceItemId,confidence
0,82442,Alpino-Webservice,CLARIAH-NL Tools,82443,Alpino-Webservice,CLARIAH-NL Tools,100.000000,True,True,False,True,high
2,75999,Antconc,CLARIN Resource Families,78153,AntConc,NaN,100.000000,True,True,False,False,high
8,64534,Cocoon,NaN,82619,COCOON,NaN,100.000000,True,False,False,False,high
9,67483,Corpkit,TAPoR,76043,CorpKit,CLARIN Resource Families,100.000000,True,True,False,False,high
10,75966,Corpus Explorer,CLARIN Resource Families,72117,CorpusExplorer,TAPoR,96.551724,True,False,False,False,high
1,67391,A.nnotate,TAPoR,67392,Annotate,TAPoR,94.117647,True,False,False,False,high
3,75999,Antconc,CLARIN Resource Families,76000,AntPConc,CLARIN Resource Families,93.333333,True,False,False,False,high
4,78153,AntConc,NaN,76000,AntPConc,CLARIN Resource Families,93.333333,True,False,False,False,high
5,15240,Automatic Transcription of Dutch Speech Recordings (MP3 file),Language Resource Switchboard,34634,Automatic Transcription of Dutch Speech Recordings (Ogg file),Language Resource Switchboard,93.220339,True,True,False,False,high
6,15240,Automatic Transcription of Dutch Speech Recordings (MP3 file),Language Resource Switchboard,15242,Automatic Transcription of Dutch Speech Recordings (Wav file),Language Resource Switchboard,93.220339,True,True,False,False,high


## 9. Create a reviewer table

The goal of this workflow is not to make automatic decisions, but to support human review. The table below highlights each likely duplicate pair and includes enough context for a curator to evaluate it, such as the titles, sources, similarity score, exact-match signals, and the assigned confidence level.


In [ ]:
review_columns = [
    "label_1",
    "label_2",
    "source_1",
    "source_2",
    "title_score",
    "same_external_id",
    "same_url",
    "same_persistentId",
    "same_sourceItemId",
    "confidence",
]

review_df = pairs_df[pairs_df["confidence"].isin(["high", "medium"])].copy()

review_df[review_columns].head(25)

,label_1,label_2,source_1,source_2,title_score,same_external_id,same_url,same_persistentId,same_sourceItemId,confidence
0,Alpino-Webservice,Alpino-Webservice,CLARIAH-NL Tools,CLARIAH-NL Tools,100.000000,True,True,False,True,high
2,Antconc,AntConc,CLARIN Resource Families,NaN,100.000000,True,True,False,False,high
8,Cocoon,COCOON,NaN,NaN,100.000000,True,False,False,False,high
9,Corpkit,CorpKit,TAPoR,CLARIN Resource Families,100.000000,True,True,False,False,high
10,Corpus Explorer,CorpusExplorer,CLARIN Resource Families,TAPoR,96.551724,True,False,False,False,high
1,A.nnotate,Annotate,TAPoR,TAPoR,94.117647,True,False,False,False,high
3,Antconc,AntPConc,CLARIN Resource Families,CLARIN Resource Families,93.333333,True,False,False,False,high
4,AntConc,AntPConc,NaN,CLARIN Resource Families,93.333333,True,False,False,False,high
5,Automatic Transcription of Dutch Speech Recordings (MP3 file),Automatic Transcription of Dutch Speech Recordings (Ogg file),Language Resource Switchboard,Language Resource Switchboard,93.220339,True,True,False,False,high
6,Automatic Transcription of Dutch Speech Recordings (MP3 file),Automatic Transcription of Dutch Speech Recordings (Wav file),Language Resource Switchboard,Language Resource Switchboard,93.220339,True,True,False,False,high


## 10. Summarize the results

This summary provides a compact overview of the sample size and the number of candidate duplicate pairs found. The values are converted to standard Python integers so the final output is clean and easy to read.


In [ ]:
summary = {
    "total_items": int(len(dup_df)),
    "candidate_pairs": int(len(pairs_df)),
    "high_confidence": int((pairs_df["confidence"] == "high").sum()),
    "medium_confidence": int((pairs_df["confidence"] == "medium").sum())
}

summary

{'total_items': 500,
 'candidate_pairs': 12,
 'high_confidence': 12,
 'medium_confidence': 0}

## Findings

From a sample of 500 tool-or-service records, the pipeline identified a small set of candidate duplicate pairs. In my run, the final summary was:

```python
{
    "total_items": 500,
    "candidate_pairs": 12,
    "high_confidence": 12,
    "medium_confidence": 0
}
```

Most of the high-confidence matches were backed by strong metadata signals, such as shared URLs, matching identifiers, or almost identical normalized labels. The medium-confidence pairs are still worth looking at, but they rely more on title similarity, so they would need a bit more careful manual review.

One of the main advantages of this approach is that it narrows down a large set of records into a small, focused list of likely duplicates. Instead of overwhelming the reviewer with raw similarity scores, it provides a clear, evidence-based shortlist that is much easier to inspect and act on.


## Limitations and reflection

This solution is intentionally kept simple and easy to understand. It works well as a lightweight prototype for identifying potential duplicates, but it’s not meant to be a fully automated system that makes final decisions on its own.

There are a few important limitations to keep in mind:

- The analysis is based on a sample of the catalogue, not the full Marketplace, so it may not capture all patterns.
- Fuzzy matching on titles can miss duplicates when the names are very different, even if other fields like descriptions or URLs clearly refer to the same resource.
- On the other hand, similar titles can sometimes belong to different but related tools, so manual review is still necessary.
- The confidence rules are based on simple thresholds and would benefit from tuning using real, validated examples.
- Since there is no labelled dataset available, the workflow does not measure precision or recall.

The current approach prioritizes interpretability over completeness, so it may miss some duplicates in favor of keeping the results easy to explain and review.

With more time, I would expand this by running the pipeline on the full dataset, introducing better filtering (blocking) strategies, comparing additional metadata fields more deeply, and exploring semantic similarity methods using embeddings. I would also create a small labelled dataset to properly evaluate how well the approach performs.
